# How good is the published evidence for QPOP?

QPOP tests ~155 drug combinations on a patient's tumour cells in the lab and predicts which combination will work.
Its papers report high accuracy. This notebook re-checks those numbers using only what the papers themselves publish.

**Real:** every count in sections 1-3 (sources in `docs/sources.md`).
**Simulated:** section 4, which asks how much of a reported accuracy can come from choosing the cutoff on the same small data set.

In [1]:
import math
import numpy as np
import plotly.graph_objects as go

def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes out of n."""
    p = k / n
    c = (p + z*z/(2*n)) / (1 + z*z/n)
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return c - h, c + h

def fisher_two_sided(a, b, c, d):
    """Fisher exact test for the 2x2 table [[a, b], [c, d]]."""
    r1, c1, n = a + b, a + c, a + b + c + d
    pr = lambda x: math.comb(c1, x) * math.comb(n - c1, r1 - x) / math.comb(n, r1)
    p0 = pr(a)
    return sum(pr(x) for x in range(max(0, r1 + c1 - n), min(r1, c1) + 1) if pr(x) <= p0 * (1 + 1e-9))

## 1. AML (Sahib et al. 2024): from 63 samples to the headline number

Counts from the paper's results section. The 86.2% accuracy is computed on the last bar only.

In [2]:
funnel = [("samples collected", 63), ("QPOP runs", 59), ("passed quality check", 55),
          ("evaluated vs outcome", 38), ("used for headline accuracy", 29)]
fig = go.Figure(go.Bar(x=[n for _, n in funnel], y=[s for s, _ in funnel], orientation="h",
                       text=[n for _, n in funnel], textposition="outside"))
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="samples", height=330,
                  title="AML: 29 of 63 samples (46%) reach the accuracy figure")
fig

## 2. AML accuracy, re-computed with honest baselines

2x2 table from the paper: 15 true positives, 3 false negatives, 10 true negatives, 1 false positive.
Baseline: predict "responds" for everyone. Kappa measures agreement beyond chance (0 = chance, 1 = perfect).

In [3]:
TP, FN, TN, FP = 15, 3, 10, 1
N_AML = TP + FN + TN + FP
metrics = {"sensitivity": (TP, TP + FN), "specificity": (TN, TN + FP), "PPV": (TP, TP + FP),
           "NPV": (TN, TN + FN), "accuracy": (TP + TN, N_AML), "baseline: everyone responds": (TP + FN, N_AML)}
for name, (k, n) in metrics.items():
    lo, hi = wilson(k, n)
    print(f"{name:28s} {k:2d}/{n:2d} = {k/n:6.1%}   95% CI {lo:6.1%} - {hi:6.1%}")
po = (TP + TN) / N_AML
pe = ((TP + FP) * (TP + FN) + (TN + FN) * (TN + FP)) / N_AML**2
kappa = (po - pe) / (1 - pe)
p_fisher = fisher_two_sided(TP, FN, FP, TN)
print(f"Cohen's kappa = {kappa:.2f}, Fisher exact p = {p_fisher:.4f}")

sensitivity                  15/18 =  83.3%   95% CI  60.8% -  94.2%
specificity                  10/11 =  90.9%   95% CI  62.3% -  98.4%
PPV                          15/16 =  93.8%   95% CI  71.7% -  98.9%
NPV                          10/13 =  76.9%   95% CI  49.7% -  91.8%
accuracy                     25/29 =  86.2%   95% CI  69.4% -  94.5%
baseline: everyone responds  18/29 =  62.1%   95% CI  44.0% -  77.3%
Cohen's kappa = 0.72, Fisher exact p = 0.0003


In [4]:
names = list(metrics)
vals = [k / n for k, n in metrics.values()]
ci = [wilson(k, n) for k, n in metrics.values()]
fig = go.Figure(go.Bar(x=names, y=vals, error_y=dict(type="data", symmetric=False,
                       array=[h - v for v, (l, h) in zip(vals, ci)], arrayminus=[v - l for v, (l, h) in zip(vals, ci)])))
fig.update_layout(yaxis=dict(title="proportion", range=[0, 1.05]), height=420,
                  title=f"AML, n = {N_AML}: wide intervals; accuracy vs the do-nothing baseline")
fig

## 3. Lymphoma (NHL) numbers

Abstract-level counts. The 2025 paper reports 74.5% accuracy on 105 evaluable cases, but no whole number out of 105 gives 74.5%
(78/105 = 74.3%, 79/105 = 75.2%). So the accuracy denominator is not 105; the abstract does not say what it is. 78/105 below is an approximation.

In [5]:
nhl = {"2022: complete response on QPOP-guided therapy": (5, 17),
       "2025: accuracy (78/105, back-calculated)": (78, 105),
       "AML 2024: accuracy": (TP + TN, N_AML)}
for name, (k, n) in nhl.items():
    lo, hi = wilson(k, n)
    print(f"{name:48s} {k:3d}/{n:3d} = {k/n:6.1%}   95% CI {lo:6.1%} - {hi:6.1%}")

2022: complete response on QPOP-guided therapy     5/ 17 =  29.4%   95% CI  13.3% -  53.1%
2025: accuracy (78/105, back-calculated)          78/105 =  74.3%   95% CI  65.2% -  81.7%
AML 2024: accuracy                                25/ 29 =  86.2%   95% CI  69.4% -  94.5%


## 4. Simulation: how much accuracy comes from tuning the cutoff on the same data?

In the AML paper the 0.65 cutoff was picked by ROC analysis on the same patients it was then scored on.
Here: two groups (responders, non-responders) whose lab scores differ by an effect size *d* (0 = the test carries no information).
For each simulated study we pick the best cutoff on the study's own patients, record the accuracy that study would report,
then score the same cutoff on 20,000 new patients. The gap is optimism. Response rate fixed at 62% as in the AML cohort.

In [6]:
rng = np.random.default_rng(0)
RESP = 18 / 29

def optimism(n, d, reps=2000):
    """Mean reported (same-data) accuracy and mean true accuracy for best-cutoff studies of size n."""
    y = rng.random((reps, n)) < RESP
    s = rng.normal(0, 1, (reps, n)) - d * y          # responders have lower viability scores
    order = np.argsort(s, axis=1)
    ys = np.take_along_axis(y, order, axis=1)
    # cutoff after position k: first k called responders
    tp = np.concatenate([np.zeros((reps, 1)), np.cumsum(ys, axis=1)], axis=1)
    fp = np.arange(n + 1)[None] - tp
    tn = (~y).sum(1, keepdims=True) - fp
    acc = (tp + tn) / n
    k = acc.argmax(1)
    reported = acc[np.arange(reps), k]
    srt = np.take_along_axis(s, order, axis=1)
    pad = np.concatenate([np.full((reps, 1), -np.inf), srt, np.full((reps, 1), np.inf)], axis=1)
    cut = (pad[np.arange(reps), k] + pad[np.arange(reps), k + 1]) / 2
    cut = np.where(np.isfinite(cut), cut, np.where(k == 0, -np.inf, np.inf))
    # true accuracy of each cutoff, exact from the normal distribution
    Phi = lambda x: 0.5 * (1 + np.vectorize(math.erf)(x / math.sqrt(2)))
    true = RESP * Phi(cut + d) + (1 - RESP) * (1 - Phi(cut))
    return reported.mean(), true.mean()

D = np.round(np.arange(0, 2.01, 0.25), 2)
NS = np.arange(10, 101, 10)
REP = np.zeros((len(NS), len(D))); TRUE = np.zeros_like(REP)
for i, n in enumerate(NS):
    for j, d in enumerate(D):
        REP[i, j], TRUE[i, j] = optimism(n, d)
j29 = [optimism(29, d) for d in D]
print("n = 29:  d   reported  true")
for d, (r, t) in zip(D, j29):
    print(f"      {d:4.2f}   {r:6.1%}  {t:6.1%}")

n = 29:  d   reported  true
      0.00    66.8%   56.1%
      0.25    69.3%   58.2%
      0.50    72.2%   61.0%
      0.75    74.8%   64.3%
      1.00    77.9%   68.4%
      1.25    81.3%   72.2%
      1.50    84.1%   75.9%
      1.75    86.7%   79.3%
      2.00    89.3%   82.5%


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=D, y=[r for r, _ in j29], mode="lines+markers", name="reported (cutoff tuned on same 29)"))
fig.add_trace(go.Scatter(x=D, y=[t for _, t in j29], mode="lines+markers", name="true, on new patients"))
fig.add_hline(y=RESP, line_dash="dot", annotation_text="everyone-responds baseline 62%")
fig.add_hline(y=25 / 29, line_dash="dash", annotation_text="AML paper 86.2%")
fig.update_layout(xaxis_title="effect size d (0 = test is useless)", yaxis_title="accuracy", height=430,
                  title="n = 29: what a tuned-cutoff study reports vs what it really delivers")
fig

### Optimism over study size and effect size (rotate it)

In [8]:
fig = go.Figure(go.Surface(x=D, y=NS, z=REP - TRUE, colorscale="Viridis",
                           colorbar=dict(title="optimism")))
fig.update_layout(scene=dict(xaxis_title="effect size d", yaxis_title="patients in study",
                             zaxis_title="reported minus true accuracy"),
                  title="Small studies with a tuned cutoff overstate accuracy", height=600)
fig

## What this says

1. **The AML signal is real, but small and selected.** Kappa 0.72 and Fisher p = 0.0003 mean QPOP did better than chance on these 29 samples.
   But the 29 are 46% of the samples collected. Deaths before treatment (8), loss to follow-up (4) and off-panel treatment (5) were removed, and those are not random exclusions.
2. **The "do nothing" baseline is 62%.** Calling every patient a responder scores 62.1%, so QPOP's 86.2% is +24 points, not +36 over a coin flip.
3. **The interval is wide.** The 95% CI for accuracy is 69%–95%. One more study of 29 patients could land anywhere in that range.
4. **The cutoff was tuned on the same patients.** In the simulation, a study of 29 patients with a tuned cutoff overstates accuracy by about 7–11 points at every effect size. A reported 86% is consistent with a true accuracy near 79%. The paper also used the first 22 samples to adjust drug doses before the 29 were scored, which adds more optimism, of a size that cannot be estimated from the paper.
5. **Small reporting problems.** The AML AUC interval is given as 0.612–1.005, but an AUC cannot exceed 1, which points to a normal-approximation interval. The NHL 74.5% does not match a denominator of 105.
6. **No independent replication.** Every QPOP paper found comes from one group, with a company (KYAN Technologies) providing the analysis software.
7. **Myeloma has no clinical QPOP data.** The 2018 myeloma paper is cell lines, mice and ex vivo samples, with no patient outcomes.

Bottom line: QPOP is a plausible lab test with early, single-group, optimistic evidence in blood cancers. It is not yet evidence that QPOP-guided treatment beats physician choice. None of the papers in `docs/sources.md` is a randomized comparison.